In [ ]:
import cv2
import numpy as np
import time
from ultralytics import YOLO

coco_model = YOLO("yolov10n.pt") 
miner_model = YOLO("best.pt")     

clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))


cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

prev_time = 0

def draw_box(img, box, label, color):
    x1, y1, x2, y2 = map(int, box[:4])
    conf = float(box[4]) if len(box) > 4 else 0.0
    text = f"{label} {conf:.2f}"
    
   
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(img, (x1, y1 - 20), (x1 + w, y1), color, -1)
    cv2.putText(img, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_enhanced = clahe.apply(l)
    enhanced_frame = cv2.cvtColor(cv2.merge((l_enhanced, a, b)), cv2.COLOR_LAB2BGR)

   
    coco_res = coco_model(
        enhanced_frame,
        classes=[0],     
        conf=0.35,
        imgsz=480,
        device="cpu",
        verbose=False
    )[0]

    
    miner_res = miner_model(
        enhanced_frame,
        conf=0.25,
        imgsz=480,
        device="cpu",
        verbose=False
    )[0]

   
    display_frame = enhanced_frame.copy()

    
    coco_count = 0
    if coco_res.boxes is not None:
        for b in coco_res.boxes.data.cpu().numpy():
            draw_box(display_frame, b, "person", (0, 255, 0))
            coco_count += 1

    
    miner_count = 0
    if miner_res.boxes is not None:
        for b in miner_res.boxes.data.cpu().numpy():
            draw_box(display_frame, b, "miner", (0, 165, 255))
            miner_count += 1

    
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
    prev_time = curr_time

    status_str = f"FPS: {int(fps)} | Persons: {coco_count} | Miners: {miner_count}"
    cv2.putText(display_frame, status_str, (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    cv2.imshow("Ensemble Detection (Base Person + Fine-tuned Miner)", display_frame)

    if cv2.waitKey(1) & 0xFF in (ord('q'), 27):
        break

cap.release()
cv2.destroyAllWindows()